# Mapperatorinator – helyi beatmap-generálás

Ez a jegyzet az [OliBomby Mapperatorinator](https://github.com/OliBomby/Mapperatorinator) modelljével készít osu! beatmapet mind a négy játékmódra. A Colab-demó helyi változata: a környezetet uv kezeli, a számítás NVIDIA GPU-n fut (RTX 3090 és 4090 is).

Az angol jegyzet: `mapperatorinator_inference_en.ipynb`.

### Használd felelősen

A mesterséges intelligencia használatát a beatmapben mindig jelezd.

### Indítás

* Fogadd el a szabályt az űrlap tetején lévő négyzettel.
* A cellákat fentről lefelé futtasd.
* Telepítés, aztán indítás. Dupla kattintás: `scripts\install.bat`, majd `scripts\start.bat`.

```powershell
.\scripts\install.ps1
.\scripts\start.ps1
```

* **Hang:** `.mp3`, `.ogg`, `.wav`, `.flac` vagy `.m4a`.
* **Referencia beatmap,** ha kell: `.osu` fájl. Stable-ben File → Open Song Folder, lazerben File → Edit Externally. In-context használatához és egy szakasz újraírásához kötelező. Ha a dal a `.osu` mellett van, a hang útvonala üresen maradhat.
* Állítsd be a stílust, majd nyomd meg a **Beatmap generálása** gombot. Az első futás a modellt a Hugging Face-ről tölti le, ez több gigabájt. Utána egy átlagos dal néhány perc.
* A **-1** ismeretlent jelent. A V32 a legjobb modell. A V30 nem tud más játékmódot, évet, címkét vagy in-contextet.

### Feladatok

* **Új beatmap:** timing, kiai, objektumok és slider velocity. Elég a hang. Az eredmény `.osz`.
* **Csak timing:** lassabb, pontosabb szuper timing. Változó BPM-hez.
* **Vendég nehézség:** egy másik nehézség ritmusát követi, a mintákat nem másolja.
* **Hitsound:** a referencia objektumai megmaradnak, a modell hitsoundot tesz rájuk.
* **Szakasz:** egy időintervallum újragenerálása, az az előtti rész megmarad.
* **Egyéni:** minden kapcsoló szerkeszthető.

A beállítások a `beallitasok.json` fájlba kerülnek.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def projekt_mappa() -> Path:
    jelolt = Path.cwd().resolve()
    for mappa in (jelolt, *jelolt.parents):
        if (mappa / "pyproject.toml").exists() and any((mappa / nev).exists() for nev in ("mapperatorinator_inference.ipynb", "mapperatorinator_inference_en.ipynb")):
            return mappa
    return Path(r"C:\Users\Zsolt\Documents\pythons\osu_ai_mapper")


PROJEKT = projekt_mappa()
REPO = PROJEKT / "Mapperatorinator"
FFMPEG_BIN = PROJEKT / "eszkozok" / "ffmpeg" / "bin"
KIMENET = PROJEKT / "kimenet"
BEALLITASOK = PROJEKT / "beallitasok.json"
FFMPEG_URL = "https://github.com/GyanD/codexffmpeg/releases/download/7.1.1/ffmpeg-7.1.1-full_build-shared.zip"

os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")


def parancs(argumentumok):
    return subprocess.run(argumentumok, check=True, text=True, capture_output=True)


def biztosit_repot():
    if (REPO / "inference.py").exists():
        return
    print("A Mapperatorinator almodul még nincs a gépen, letöltöm...")
    parancs(["git", "-C", str(PROJEKT), "submodule", "update", "--init", "--depth", "1"])


def biztosit_ffmpeg():
    if (FFMPEG_BIN / "ffmpeg.exe").exists() and (FFMPEG_BIN / "ffprobe.exe").exists():
        return
    import io
    import urllib.request
    import zipfile

    print("Az FFmpeg 7.1 shared változatát letöltöm (a hangfájlok olvasásához kell)...")
    FFMPEG_BIN.parent.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(FFMPEG_URL) as valasz:
        csomag = zipfile.ZipFile(io.BytesIO(valasz.read()))
    cel = PROJEKT / "eszkozok"
    csomag.extractall(cel)
    kibontott = next(cel.glob("ffmpeg-*"))
    if FFMPEG_BIN.parent.exists():
        raise RuntimeError(f"Az FFmpeg mappa már létezik, de hiányos: {FFMPEG_BIN.parent}")
    kibontott.rename(cel / "ffmpeg")


def beallit_utvonalakat():
    if str(FFMPEG_BIN) not in os.environ.get("PATH", ""):
        os.environ["PATH"] = str(FFMPEG_BIN) + os.pathsep + os.environ.get("PATH", "")
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(str(FFMPEG_BIN))
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))


biztosit_repot()
biztosit_ffmpeg()
KIMENET.mkdir(exist_ok=True)
beallit_utvonalakat()

if sys.maxsize <= 2**32:
    raise RuntimeError(
        "32 bites Python fut. A CUDA-s PyTorch csak 64 bites interpreteren működik.\n"
        "Indítás: .\\scripts\\start.ps1"
    )

venv = (PROJEKT / ".venv").resolve()
if Path(sys.prefix).resolve() != venv:
    print("FIGYELEM: nem a projekt uv környezete fut, ezért a csomagok hiányozhatnak.")
    print("PowerShellben, a projekt mappájából")
    print("  .\\scripts\\start.ps1")
    print(f"Vagy ezt a kernelt válaszd: {venv / 'Scripts' / 'python.exe'}")

import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", SyntaxWarning)
    from pydub import AudioSegment

AudioSegment.converter = str(FFMPEG_BIN / "ffmpeg.exe")
AudioSegment.ffprobe = str(FFMPEG_BIN / "ffprobe.exe")

import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nincs CUDA"
print(f"Python: {sys.version.split()[0]}  ({'64 bites' if sys.maxsize > 2**32 else '32 bites'})")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {gpu}")
if not torch.cuda.is_available():
    print("CUDA nélkül a generálás nagyon lassú, vagy el sem indul. Ellenőrizd a drivert.")

import inference  # noqa: F401  a ConfigStore a compose előtt kell
from osuT5.osuT5.event import ContextType

verzio = parancs(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"]).stdout.strip()
ffmpeg_verzio = parancs([str(FFMPEG_BIN / "ffmpeg.exe"), "-version"]).stdout.splitlines()[0]
print(f"Mapperatorinator: {verzio}")
print(f"FFmpeg: {ffmpeg_verzio}")
print(f"Kimeneti mappa: {KIMENET}")
print("A környezet kész. Futtasd a következő cellát, és töltsd ki az űrlapot.")

KORNYEZET_KESZ = True
OSU_SONGS = Path.home() / "AppData" / "Local" / "osu!" / "Songs"


In [ ]:
import json
import os
import sys
import time
import traceback
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

if not globals().get("KORNYEZET_KESZ"):
    raise RuntimeError("Először futtasd a fölötte lévő cellát.")

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from osuT5.osuT5.event import ContextType

CIMKE_FAJL = REPO / "datasets" / "tags_2026.json"
NYERS_CIMKEK = json.loads(CIMKE_FAJL.read_text(encoding="utf-8"))["tags"]
ISMERT = {}
for tag in NYERS_CIMKEK:
    ISMERT.setdefault(tag["name"], tag)

MAGYAR = {
    "style/messy": "rendezetlen, zsúfolt stílus",
    "expression/simple": "egyszerű",
    "expression/chaotic": "kaotikus",
    "style/geometric": "geometrikus",
    "style/freeform": "szabad formájú",
    "style/grid snap": "rácsra igazított",
    "style/hexgrid": "hatszögrács",
    "style/symmetrical": "szimmetrikus",
    "gimmick/playfield constraint": "a pálya csak egy része használt",
    "expression/old-style revival": "régi mappingstílus",
    "expression/repetition": "ismétlődő minták",
    "expression/progression": "fokozatos nehezedés",
    "expression/high contrast": "nagy kontraszt a részek között",
    "expression/improvisation": "improvizáció",
    "skillset/tech": "tech",
    "tech/slider tech": "slider tech",
    "tech/finger control": "finger control",
    "sliders/complex sv": "bonyolult slider velocity",
    "skillset/jumps": "ugrások",
    "jumps/sharp": "éles szögű ugrások",
    "jumps/wide": "széles szögű ugrások",
    "jumps/linear": "egyenes vonalú ugrások",
    "tech/aim control": "aim control",
    "streams/flow aim": "flow aim",
    "streams/bursts": "burstök",
    "skillset/streams": "streamek",
    "streams/spaced streams": "ritkított streamek",
    "streams/stamina": "hosszú, fárasztó streamek",
    "streams/cutstreams": "kihagyásos streamek",
    "skillset/reading": "olvasás",
    "reading/visually dense": "vizuálisan sűrű",
    "reading/overlaps": "átfedések",
    "skillset/precision": "precíziós aim",
    "skillset/alt": "alternating",
    "meta/collab": "több mapper",
    "meta/multi-song": "több dal egy mapban",
    "meta/variable timing": "változó timing",
    "meta/time signatures": "szokatlan ütemmutató",
    "style/clean": "tiszta, rendezett",
    "sliders/complex slidershapes": "bonyolult sliderformák",
    "gimmick/aspire": "aspire",
    "style/distance snap": "distance snap",
    "context/mapping contest": "mapping verseny",
    "context/tournament custom": "tournament custom",
    "tech/complex snap": "szokatlan snapek",
    "expression/iNiS-style": "iNiS stílus",
    "gimmick/2B": "egyszerre több objektum (2B)",
    "expression/playfield usage": "tudatos pályahasználat",
    "gimmick/tag": "tag multi",
    "expression/difficulty spike": "hirtelen nehézségugrás",
    "style/avant-garde": "avantgárd",
    "additions/keysounds": "keysound dallam",
    "gimmick/storyboard": "storyboard változtat a játékon",
    "sliders/low sv": "alacsony SV",
    "sliders/high sv": "magas SV",
    "meta/mega collab": "mega collab",
    "gimmick/slider only": "csak slider",
    "gimmick/circle only": "csak kör",
    "additions/custom skin": "egyedi skin",
    "gimmick/video": "videókövetés",
    "reading/perfect stacks": "tökéletes stackek",
    "gimmick/ninja spinners": "nagyon rövid spinnerek",
    "meta/accelerating bpm": "gyorsuló BPM",
    "context/custom song": "a maphez írt dal",
    "gimmick/mirrored": "tükrözött minták",
    "style/vocal": "vokálkövetés",
    "style/tnt": "Taiko no Tatsujin stílus",
    "streams/speed": "nagyon gyors tappolás",
    "style/double bpm": "dupla BPM",
    "style/finisher-heavy": "sok finisher",
    "style/mono-heavy": "sok mono",
    "gimmick/reversed": "fordított minták",
    "gimmick/yellow notes": "nagyon rövid sárga noték",
    "gimmick/barlines": "barline-ok",
    "gimmick/memory": "memorizálós",
    "style/taikosu": "osu! és taiko együtt",
    "meta/swing": "swing ritmus",
    "style/convert": "convert stílus",
    "additions/hitsounds": "egyedi hitsoundok",
    "tech/flow": "flow",
    "tech/antiflow": "antiflow",
    "tech/jump": "ugrás / dash",
    "gimmick/no hyperdashes": "hyperdash nélkül",
    "gimmick/long sliders": "hosszú sliderek",
    "gimmick/spinner-heavy": "sok spinner",
    "tech/wiggles": "gyors irányváltások",
    "tech/hyperwalks": "hyperwalk",
    "gimmick/dodge the beat": "az objektumokat kerülni kell",
    "style/jumpstream": "jumpstream",
    "style/handstream": "handstream",
    "style/quadstream": "quadstream",
    "style/chordjack": "chordjack",
    "style/longjack": "longjack",
    "skillset/speedjack": "speedjack",
    "skillset/wristjack": "wristjack",
    "style/chordstream": "chordstream",
    "gimmick/delay": "delay",
    "style/mixed rice": "vegyes rice",
    "style/LN coordination": "LN koordináció",
    "style/LN release": "LN elengedés",
    "gimmick/LN inverse": "fordított LN",
    "style/LN density": "sűrű LN",
    "style/LN mixed ": "vegyes LN",
    "style/generic hybrid": "egyszerű rice + LN",
    "tech/technical hybrid": "technikai rice + LN",
    "style/tiebreaker": "tiebreaker",
    "style/o2jam": "o2jam",
    "style/N+1": "N+1",
    "style/dump": "dump",
    "jumps/squares": "négyzetes ugrások",
    "jumps/triangles": "háromszög ugrások",
    "jumps/cross-screen": "keresztképernyős ugrások",
    "jumps/stamina": "hosszú ugrássorok",
    "skillset/gimmick": "gimmick",
    "jumps/stars": "csillag alakú ugrások",
    "jumps/back and forth": "oda-vissza ugrások",
    "jumps/freeform": "szabad ugrások",
    "streams/doubles": "dupla hangok",
    "streams/quads": "négyes hangok",
    "expression/inspo": "inspiráció egy másik mapből",
    "expression/conceptual": "konceptuális",
    "additions/combo colours": "combo színek",
}

FELADATOK = {
    "uj": {
        "leiras": "A dalból új beatmap készül: timing, kiai, objektumok és slider velocity. Elég a hangfájl.",
        "kontextus": "[NONE]",
        "kimenet": "[TIMING,MAP,SV]",
        "hozzaad": False,
        "osz": True,
        "super_timing": False,
        "hitsounded": None,
        "referencia": False,
        "szakasz": False,
    },
    "timing": {
        "leiras": "Csak timing készül, a lassabb és pontosabb szuper timinggal. Változó BPM-hez ezt használd.",
        "kontextus": "[NONE]",
        "kimenet": "[TIMING]",
        "hozzaad": False,
        "osz": False,
        "super_timing": True,
        "hitsounded": None,
        "referencia": False,
        "szakasz": False,
    },
    "vendeg": {
        "leiras": "Vendég nehézség: a referencia ritmusát és timingjét követi, de nem másolja a mintákat. Kell egy .osu ugyanabból a szettből.",
        "kontextus": "[GD,TIMING,KIAI]",
        "kimenet": "[MAP]",
        "hozzaad": False,
        "osz": False,
        "super_timing": False,
        "hitsounded": None,
        "referencia": True,
        "szakasz": False,
    },
    "hitsound": {
        "leiras": "A referencia objektumai megmaradnak, a modell hitsoundot tesz rájuk. Kell a .osu.",
        "kontextus": "[NO_HS,TIMING,KIAI]",
        "kimenet": "[MAP]",
        "hozzaad": False,
        "osz": False,
        "super_timing": False,
        "hitsounded": True,
        "referencia": True,
        "szakasz": False,
    },
    "szakasz": {
        "leiras": "Egy időintervallum újragenerálása, a szakasz előtti objektumok megmaradnak. Add meg a kezdetet és a véget ezredmásodpercben, és a .osu fájlt.",
        "kontextus": "[TIMING,KIAI]",
        "kimenet": "[MAP]",
        "hozzaad": True,
        "osz": False,
        "super_timing": False,
        "hitsounded": None,
        "referencia": True,
        "szakasz": True,
    },
    "egyeni": {
        "leiras": "Minden kapcsoló szerkeszthető. A −1 ismeretlent vagy a referencia map értékét jelenti.",
        "referencia": False,
        "szakasz": True,
    },
}

KONTEXTUSOK = [
    ("Nincs plusz kontextus", "[NONE]"),
    ("Kész timing", "[TIMING]"),
    ("Timing és kiai", "[TIMING,KIAI]"),
    ("A referencia objektumai", "[TIMING,KIAI,MAP]"),
    ("Másik nehézség ritmusa", "[GD,TIMING,KIAI]"),
    ("Objektumok hitsound nélkül", "[NO_HS,TIMING,KIAI]"),
]
KIMENETEK = [
    ("Timing, objektumok és SV", "[TIMING,MAP,SV]"),
    ("Csak az objektumok", "[MAP]"),
    ("Csak timing", "[TIMING]"),
]


def cimke_nev(szoveg):
    szoveg = (szoveg or "").strip()
    if not szoveg:
        return ""
    if " — " in szoveg:
        szoveg = szoveg.split(" — ", 1)[1].strip()
    if szoveg in ISMERT:
        return szoveg
    if f"{szoveg} " in ISMERT:
        return f"{szoveg} "
    return szoveg


def cimke_opciok(gamemode, mind):
    lathato = []
    mar = set()
    for nev, tag in ISMERT.items():
        if nev in mar:
            continue
        szabaly = tag.get("ruleset_id")
        if not mind and szabaly not in (None, gamemode):
            continue
        mar.add(nev)
        lathato.append((MAGYAR.get(nev, nev), nev))
    lathato.sort(key=lambda par: par[0].casefold())
    return ["", *[f"{felirat} — {nev}" for felirat, nev in lathato]]


def szam_vagy_ures(ertek, egesz=False):
    if ertek is None or ertek == -1:
        return None
    return int(ertek) if egesz else float(ertek)


def szoveg_vagy_ures(ertek):
    ertek = (ertek or "").strip()
    return ertek or None


def kontextus_listaba(szoveg):
    belso = szoveg.strip()[1:-1]
    if not belso or belso.upper() == "NONE":
        return [ContextType.NONE]
    return [ContextType[resz.strip().upper()] for resz in belso.split(",") if resz.strip()]


def cimkek_listaja(dobozok):
    eredmeny = []
    for doboz in dobozok:
        nev = cimke_nev(doboz.value)
        if not nev:
            continue
        if nev not in ISMERT:
            raise ValueError(f"Ismeretlen címke: {doboz.value}")
        if nev not in eredmeny:
            eredmeny.append(nev)
    return eredmeny or None


def talloz(cel, cim, fajlok=None, mappa=False):
    import tkinter as tk
    from tkinter import filedialog

    gyoker = tk.Tk()
    gyoker.withdraw()
    gyoker.attributes("-topmost", True)
    kezdo = None
    if cel.value and Path(cel.value).exists():
        ut = Path(cel.value)
        kezdo = str(ut if ut.is_dir() else ut.parent)
    elif OSU_SONGS.exists():
        kezdo = str(OSU_SONGS)
    argumentumok = {"title": cim}
    if kezdo:
        argumentumok["initialdir"] = kezdo
    if mappa:
        valasztott = filedialog.askdirectory(**argumentumok)
    else:
        argumentumok["filetypes"] = fajlok
        valasztott = filedialog.askopenfilename(**argumentumok)
    gyoker.destroy()
    if valasztott:
        cel.value = valasztott


def mezo(cim, vezerlo, sugo=""):
    sorok = [widgets.HTML(f"<b>{cim}</b>")]
    if sugo:
        sorok.append(widgets.HTML(f"<span style='color:#555'>{sugo}</span>"))
    sorok.append(vezerlo)
    return widgets.VBox(sorok, layout=widgets.Layout(width="360px", margin="0 16px 12px 0"))


def sor(*mezok):
    return widgets.Box(list(mezok), layout=widgets.Layout(display="flex", flex_flow="row wrap"))


def konfig_betoltese(nev):
    if GlobalHydra.instance().is_initialized():
        GlobalHydra.instance().clear()
    config_dir = (REPO / "configs" / "inference").resolve().as_posix()
    with initialize_config_dir(version_base="1.1", config_dir=config_dir):
        return compose(config_name=nev)


elfogadom = widgets.Checkbox(value=False, description="Elfogadom", indent=False)
feladat = widgets.Dropdown(
    options=[
        ("Új beatmap", "uj"),
        ("Csak timing", "timing"),
        ("Vendég nehézség", "vendeg"),
        ("Hitsound egy kész mapra", "hitsound"),
        ("Szakasz újragenerálása", "szakasz"),
        ("Egyéni", "egyeni"),
    ],
    value="uj",
    layout=widgets.Layout(width="460px"),
)
feladat_szoveg = widgets.HTML()

hang = widgets.Text(placeholder=r"C:\...\dal.mp3", layout=widgets.Layout(width="520px"))
beatmap = widgets.Text(placeholder=r"C:\...\nehezseg.osu  (nem mindig kötelező)", layout=widgets.Layout(width="520px"))
kimenet = widgets.Text(value=str(KIMENET), layout=widgets.Layout(width="520px"))
hatter = widgets.Text(placeholder="háttérkép, üresen maradhat", layout=widgets.Layout(width="520px"))

modell = widgets.Dropdown(
    options=[
        ("V32 – ajánlott, mind a négy mód", "v32"),
        ("V32-mini – kisebb és gyorsabb", "v32-mini"),
        ("V31", "v31"),
        ("V30 – csak osu!standard", "v30"),
        ("V29", "v29"),
    ],
    value="v32",
    layout=widgets.Layout(width="460px"),
)
jatekmod = widgets.Dropdown(
    options=[("osu!standard", 0), ("osu!taiko", 1), ("osu!catch", 2), ("osu!mania", 3)],
    value=0,
    layout=widgets.Layout(width="460px"),
)
nehezseg = widgets.FloatText(value=5.0, layout=widgets.Layout(width="140px"))
mapper_id = widgets.IntText(value=-1, layout=widgets.Layout(width="140px"))
ev = widgets.IntText(value=2023, layout=widgets.Layout(width="140px"))
lora = widgets.Text(placeholder="Hugging Face repo or local folder, optional", layout=widgets.Layout(width="520px"))
minden_cimke = widgets.Checkbox(value=False, description="Minden mód címkéje", indent=False)

pozitiv = [widgets.Combobox(placeholder="keress", ensure_option=False, layout=widgets.Layout(width="340px")) for _ in range(3)]
negativ = [widgets.Combobox(placeholder="works when cfg > 1", ensure_option=False, layout=widgets.Layout(width="340px")) for _ in range(3)]

hp = widgets.FloatText(value=5, layout=widgets.Layout(width="120px"))
cs = widgets.FloatText(value=4, layout=widgets.Layout(width="120px"))
od = widgets.FloatText(value=9, layout=widgets.Layout(width="120px"))
ar = widgets.FloatText(value=8, layout=widgets.Layout(width="120px"))
sv = widgets.FloatText(value=1.4, layout=widgets.Layout(width="120px"))
slider_tick = widgets.FloatText(value=1, layout=widgets.Layout(width="120px"))
billentyu = widgets.IntSlider(value=4, min=1, max=18, step=1, layout=widgets.Layout(width="340px"))
hold = widgets.FloatText(value=-1, layout=widgets.Layout(width="120px"))
scroll = widgets.FloatText(value=-1, layout=widgets.Layout(width="120px"))
hitsounded = widgets.Checkbox(value=True, description="Hitsound (főleg mania)", indent=False)

kontextus = widgets.Dropdown(options=KONTEXTUSOK, value="[NONE]", layout=widgets.Layout(width="340px"))
kimenet_tipus = widgets.Dropdown(options=KIMENETEK, value="[TIMING,MAP,SV]", layout=widgets.Layout(width="340px"))
hozzaad = widgets.Checkbox(value=False, description="Hozzáírás a referenciához", indent=False)
osz = widgets.Checkbox(value=True, description=".osz csomag", indent=False)
felulir = widgets.Checkbox(value=False, description="Eredeti .osu felülírása", indent=False)
kezdet = widgets.IntText(value=-1, layout=widgets.Layout(width="140px"))
veg = widgets.IntText(value=-1, layout=widgets.Layout(width="140px"))
cfg = widgets.FloatSlider(value=1.0, min=1.0, max=5.0, step=0.1, readout_format=".1f", layout=widgets.Layout(width="340px"))
homerseklet = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.01, readout_format=".2f", layout=widgets.Layout(width="340px"))
seed = widgets.IntText(value=-1, layout=widgets.Layout(width="140px"))
timing_tures = widgets.IntSlider(value=20, min=0, max=100, step=1, layout=widgets.Layout(width="340px"))
super_timing = widgets.Checkbox(value=False, description="Szuper timing (lassabb, pontosabb)", indent=False)
sugarak = widgets.IntSlider(value=2, min=1, max=4, step=1, layout=widgets.Layout(width="340px"))
iteraciok = widgets.IntSlider(value=20, min=1, max=40, step=1, layout=widgets.Layout(width="340px"))
bpm_kuszob = widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.1, readout_format=".1f", layout=widgets.Layout(width="340px"))

cim = widgets.Text(placeholder="üresen: a referenciából vagy Unknown Title", layout=widgets.Layout(width="340px"))
eloado = widgets.Text(placeholder="üresen: a referenciából vagy Unknown Artist", layout=widgets.Layout(width="340px"))
keszito = widgets.Text(value="Mapperatorinator", layout=widgets.Layout(width="340px"))
verzio = widgets.Text(value="AI", layout=widgets.Layout(width="340px"))
forras = widgets.Text(placeholder="anime, game, album…", layout=widgets.Layout(width="340px"))

gomb = widgets.Button(
    description="Generálás",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="260px", height="42px"),
)
naplo = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px", max_height="420px", overflow_y="auto"))


def frissit_cimkek(*_):
    opciok = cimke_opciok(int(jatekmod.value), mind=minden_cimke.value)
    ervenyes = {cimke_nev(opcio): opcio for opcio in opciok if opcio}
    for doboz in (*pozitiv, *negativ):
        nev = cimke_nev(doboz.value)
        doboz.options = opciok
        doboz.value = ervenyes.get(nev, "")


def zarolas():
    egyeni = feladat.value == "egyeni"
    adat = FELADATOK[feladat.value]
    feladat_szoveg.value = f"<p style='margin:4px 0 8px 0'>{adat['leiras']}</p>"
    for vezerlo in (kontextus, kimenet_tipus, hozzaad, osz, super_timing):
        vezerlo.disabled = not egyeni
    szakasz = egyeni or adat.get("szakasz", False)
    kezdet.disabled = not szakasz
    veg.disabled = not szakasz
    if not egyeni and adat.get("hitsounded") is not None:
        hitsounded.value = adat["hitsounded"]
        hitsounded.disabled = True
    else:
        hitsounded.disabled = False
    felulir.disabled = not (egyeni or hozzaad.value)
    if not hozzaad.value:
        felulir.value = False


def alkalmaz_feladatot(*_):
    if feladat.value == "egyeni":
        zarolas()
        return
    adat = FELADATOK[feladat.value]
    kontextus.value = adat["kontextus"]
    kimenet_tipus.value = adat["kimenet"]
    hozzaad.value = adat["hozzaad"]
    osz.value = adat["osz"]
    super_timing.value = adat["super_timing"]
    felulir.value = False
    zarolas()


def allapot():
    return {
        "verzio": 1,
        "elfogadom": elfogadom.value,
        "feladat": feladat.value,
        "hang": hang.value,
        "beatmap": beatmap.value,
        "kimenet": kimenet.value,
        "hatter": hatter.value,
        "modell": modell.value,
        "jatekmod": jatekmod.value,
        "nehezseg": nehezseg.value,
        "mapper_id": mapper_id.value,
        "ev": ev.value,
        "lora": lora.value,
        "minden_cimke": minden_cimke.value,
        "pozitiv": [doboz.value for doboz in pozitiv],
        "negativ": [doboz.value for doboz in negativ],
        "hp": hp.value,
        "cs": cs.value,
        "od": od.value,
        "ar": ar.value,
        "sv": sv.value,
        "slider_tick": slider_tick.value,
        "billentyu": billentyu.value,
        "hold": hold.value,
        "scroll": scroll.value,
        "hitsounded": hitsounded.value,
        "kontextus": kontextus.value,
        "kimenet_tipus": kimenet_tipus.value,
        "hozzaad": hozzaad.value,
        "osz": osz.value,
        "felulir": felulir.value,
        "kezdet": kezdet.value,
        "veg": veg.value,
        "cfg": cfg.value,
        "homerseklet": homerseklet.value,
        "seed": seed.value,
        "timing_tures": timing_tures.value,
        "super_timing": super_timing.value,
        "sugarak": sugarak.value,
        "iteraciok": iteraciok.value,
        "bpm_kuszob": bpm_kuszob.value,
        "cim": cim.value,
        "eloado": eloado.value,
        "keszito": keszito.value,
        "diff_verzio": verzio.value,
        "forras": forras.value,
    }


def ertek_beallit(vezerlo, ertek):
    if ertek is not None:
        vezerlo.value = ertek


def betolt():
    if not BEALLITASOK.exists():
        frissit_cimkek()
        alkalmaz_feladatot()
        return
    adat = json.loads(BEALLITASOK.read_text(encoding="utf-8"))
    ertek_beallit(elfogadom, adat.get("elfogadom"))
    ertek_beallit(feladat, adat.get("feladat"))
    ertek_beallit(hang, adat.get("hang"))
    ertek_beallit(beatmap, adat.get("beatmap"))
    ertek_beallit(kimenet, adat.get("kimenet"))
    ertek_beallit(hatter, adat.get("hatter"))
    ertek_beallit(modell, adat.get("modell"))
    ertek_beallit(jatekmod, adat.get("jatekmod"))
    ertek_beallit(nehezseg, adat.get("nehezseg"))
    ertek_beallit(mapper_id, adat.get("mapper_id"))
    ertek_beallit(ev, adat.get("ev"))
    ertek_beallit(lora, adat.get("lora"))
    ertek_beallit(minden_cimke, adat.get("minden_cimke"))
    frissit_cimkek()
    for doboz, mentett in zip(pozitiv, adat.get("pozitiv", [])):
        doboz.value = mentett
    for doboz, mentett in zip(negativ, adat.get("negativ", [])):
        doboz.value = mentett
    for vezerlo, kulcs in (
        (hp, "hp"), (cs, "cs"), (od, "od"), (ar, "ar"), (sv, "sv"), (slider_tick, "slider_tick"),
        (billentyu, "billentyu"), (hold, "hold"), (scroll, "scroll"), (hitsounded, "hitsounded"),
        (kontextus, "kontextus"), (kimenet_tipus, "kimenet_tipus"), (hozzaad, "hozzaad"),
        (osz, "osz"), (felulir, "felulir"), (kezdet, "kezdet"), (veg, "veg"), (cfg, "cfg"),
        (homerseklet, "homerseklet"), (seed, "seed"), (timing_tures, "timing_tures"),
        (super_timing, "super_timing"), (sugarak, "sugarak"), (iteraciok, "iteraciok"),
        (bpm_kuszob, "bpm_kuszob"), (cim, "cim"), (eloado, "eloado"), (keszito, "keszito"),
        (verzio, "diff_verzio"), (forras, "forras"),
    ):
        ertek_beallit(vezerlo, adat.get(kulcs))
    zarolas()


def mentes():
    BEALLITASOK.write_text(json.dumps(allapot(), ensure_ascii=False, indent=2), encoding="utf-8")


class Naplo:
    encoding = "utf-8"

    def __init__(self, eredeti):
        self.eredeti = eredeti
        self._darab = ""
        self._utolso = 0.0

    def write(self, szoveg):
        if not szoveg:
            return
        self.eredeti.write(szoveg)
        self._darab += szoveg
        if "\r" not in self._darab and "\n" not in self._darab:
            return
        sorok = self._darab.replace("\r", "\n").split("\n")
        self._darab = sorok[-1]
        most = time.monotonic()
        for sor in sorok[:-1]:
            sor = sor.strip()
            if not sor:
                continue
            folyamat = "%" in sor or "it/s" in sor or "MB/s" in sor
            if folyamat and most - self._utolso < 1.0:
                continue
            if folyamat:
                self._utolso = most
            try:
                naplo.append_stdout(sor + "\n")
            except Exception:
                pass

    def flush(self):
        if self._darab.strip():
            try:
                naplo.append_stdout(self._darab.strip() + "\n")
            except Exception:
                pass
            self._darab = ""
        self.eredeti.flush()

    def isatty(self):
        return True

    def writable(self):
        return True

    def __getattr__(self, nev):
        return getattr(self.eredeti, nev)


def generalas(_=None):
    gomb.disabled = True
    regi_felirat = gomb.description
    gomb.description = "Folyamatban"
    naplo.clear_output(wait=True)
    eredeti_konyvtar = os.getcwd()
    eredeti_ki = sys.stdout
    eredeti_be = sys.stderr
    sys.stdout = Naplo(eredeti_ki)
    sys.stderr = Naplo(eredeti_be)
    try:
        mentes()
        if not elfogadom.value:
            raise RuntimeError("A generálás előtt fogadd el a szabályt.")

        hang_ut = szoveg_vagy_ures(hang.value)
        beatmap_ut = szoveg_vagy_ures(beatmap.value)
        kimenet_ut = szoveg_vagy_ures(kimenet.value) or str(KIMENET)
        if not hang_ut and not beatmap_ut:
            raise RuntimeError("Adj meg egy hangfájlt, vagy egy .osu-t, ami mellett ott a dal.")
        if hang_ut and not Path(hang_ut).is_file():
            raise RuntimeError(f"A hangfájl nem található: {hang_ut}")
        if beatmap_ut and not Path(beatmap_ut).is_file():
            raise RuntimeError(f"A referencia .osu nem található: {beatmap_ut}")
        if beatmap_ut and Path(beatmap_ut).suffix.lower() != ".osu":
            raise RuntimeError("A referenciának .osu fájlnak kell lennie.")
        if hang_ut and Path(hang_ut).suffix.lower() not in {".mp3", ".ogg", ".wav", ".flac", ".m4a"}:
            raise RuntimeError("A hang legyen mp3, ogg, wav, flac vagy m4a.")
        kezdo_ms = szam_vagy_ures(kezdet.value, egesz=True)
        veg_ms = szam_vagy_ures(veg.value, egesz=True)
        if kezdo_ms is not None and veg_ms is not None and kezdo_ms >= veg_ms:
            raise RuntimeError("A szakasz vége később legyen, mint a kezdete.")
        if felulir.value and not beatmap_ut:
            raise RuntimeError("Felülíráshoz kell a referencia .osu.")

        kontextus_ertek = kontextus_listaba(kontextus.value)
        kimenet_ertek = kontextus_listaba(kimenet_tipus.value)
        if modell.value in ("v32", "v32-mini") and ContextType.KIAI in kimenet_ertek:
            print("Figyelem: a V32 nem generál külön kiai sávot, a kimenet timing, objektumok és SV lesz.")
            kimenet_ertek = [elem for elem in kimenet_ertek if elem != ContextType.KIAI] or [ContextType.TIMING, ContextType.MAP, ContextType.SV]
        cimke_ertek = cimkek_listaja(pozitiv)
        negativ_ertek = cimkek_listaja(negativ)
        if modell.value == "v30":
            if int(jatekmod.value) != 0:
                raise RuntimeError("A V30 csak osu!standard mapot tud.")
            tiltott = {ContextType.KIAI, ContextType.MAP, ContextType.SV}
            if any(elem in tiltott for elem in kontextus_ertek):
                print("Figyelem: a V30 a KIAI, MAP és SV kontextust figyelmen kívül hagyja.")
                kontextus_ertek = [elem for elem in kontextus_ertek if elem not in tiltott] or [ContextType.NONE]
            if kimenet_ertek != [ContextType.MAP]:
                print("Figyelem: a V30 csak objektumokat tud írni, a kimenet csak MAP lesz.")
                kimenet_ertek = [ContextType.MAP]
            if cimke_ertek or negativ_ertek:
                print("Figyelem: a V30 nem használ címkéket, ezek kimaradnak.")
                cimke_ertek = None
                negativ_ertek = None
            if super_timing.value:
                print("Figyelem: a V30 szuper timingja nagyon lassú.")
        if negativ_ertek and cfg.value <= 1:
            print("Figyelem: a negatív címkék csak akkor hatnak, ha a cfg nagyobb 1-nél.")
        kell_referencia = bool(hozzaad.value) or any(
            elem in kontextus_ertek
            for elem in (
                ContextType.TIMING,
                ContextType.KIAI,
                ContextType.MAP,
                ContextType.SV,
                ContextType.GD,
                ContextType.NO_HS,
            )
        )
        if kell_referencia and not beatmap_ut:
            raise RuntimeError("Ehhez a beállításhoz kell egy referencia .osu.")

        Path(kimenet_ut).mkdir(parents=True, exist_ok=True)
        conf = konfig_betoltese(modell.value)
        conf.audio_path = hang_ut or ""
        conf.output_path = kimenet_ut
        conf.beatmap_path = beatmap_ut or ""
        conf.gamemode = int(jatekmod.value)
        conf.difficulty = szam_vagy_ures(nehezseg.value)
        conf.mapper_id = szam_vagy_ures(mapper_id.value, egesz=True)
        conf.year = szam_vagy_ures(ev.value, egesz=True)
        conf.lora_path = szoveg_vagy_ures(lora.value)
        conf.hitsounded = bool(hitsounded.value)
        conf.hp_drain_rate = szam_vagy_ures(hp.value)
        conf.circle_size = szam_vagy_ures(cs.value)
        conf.overall_difficulty = szam_vagy_ures(od.value)
        conf.approach_rate = szam_vagy_ures(ar.value)
        conf.slider_multiplier = szam_vagy_ures(sv.value)
        conf.slider_tick_rate = szam_vagy_ures(slider_tick.value)
        conf.keycount = int(billentyu.value)
        conf.hold_note_ratio = szam_vagy_ures(hold.value)
        conf.scroll_speed_ratio = szam_vagy_ures(scroll.value)
        conf.descriptors = cimke_ertek
        conf.negative_descriptors = negativ_ertek
        conf.export_osz = bool(osz.value)
        conf.add_to_beatmap = bool(hozzaad.value)
        conf.overwrite_reference_beatmap = bool(felulir.value)
        conf.start_time = kezdo_ms
        conf.end_time = veg_ms
        conf.in_context = kontextus_ertek
        conf.output_type = kimenet_ertek
        conf.cfg_scale = float(cfg.value)
        conf.temperature = float(homerseklet.value)
        conf.seed = szam_vagy_ures(seed.value, egesz=True)
        conf.timing_leniency = int(timing_tures.value)
        conf.super_timing = bool(super_timing.value)
        conf.timer_num_beams = int(sugarak.value)
        conf.timer_iterations = int(iteraciok.value)
        conf.timer_bpm_threshold = float(bpm_kuszob.value)
        conf.use_server = False
        conf.title = szoveg_vagy_ures(cim.value)
        conf.artist = szoveg_vagy_ures(eloado.value)
        conf.creator = szoveg_vagy_ures(keszito.value)
        conf.version = szoveg_vagy_ures(verzio.value)
        conf.source = szoveg_vagy_ures(forras.value)
        conf.background = szoveg_vagy_ures(hatter.value)

        print("A generálás elindult. Az első alkalommal a modell letöltése több gigabájt, utána egy dal jellemzően néhány perc.")
        if felulir.value:
            print("Az eredeti .osu felül lesz írva.")
        os.chdir(REPO)
        eredmeny = inference.main(conf)
        if not isinstance(eredmeny, tuple) or len(eredmeny) != 2:
            raise RuntimeError(f"A generálás nem adott vissza fájlt: {eredmeny!r}")
        utvonal = Path(eredmeny[1])
        print(f"Kész: {utvonal}")
        if utvonal.suffix.lower() == ".osz":
            print("Az .osz fájlra duplán kattintva az osu! importálja a beatmapet.")
        else:
            print("A .osu fájlt másold a dal mappájába, a hang mellé.")
        print("A beatmap leírásában jelöld, hogy mesterséges intelligencia készítette.")
        try:
            os.startfile(utvonal.parent)
        except OSError as hiba:
            print(f"A mappát nem sikerült megnyitni: {hiba}")
    except Exception as hiba:
        print(f"Hiba: {hiba}")
        if not isinstance(hiba, RuntimeError):
            traceback.print_exc()
    finally:
        sys.stdout = eredeti_ki
        sys.stderr = eredeti_be
        os.chdir(eredeti_konyvtar)
        gomb.disabled = False
        gomb.description = regi_felirat


gomb.on_click(generalas)
hang_gomb = widgets.Button(description="Tallózás", icon="folder")
beatmap_gomb = widgets.Button(description="Tallózás", icon="folder")
kimenet_gomb = widgets.Button(description="Tallózás", icon="folder")
hatter_gomb = widgets.Button(description="Tallózás", icon="folder")
hang_gomb.on_click(lambda _: talloz(hang, "Hangfájl", [("Hang", "*.mp3 *.ogg *.wav *.flac *.m4a"), ("Minden fájl", "*.*")]))
beatmap_gomb.on_click(lambda _: talloz(beatmap, "Referencia beatmap", [("osu! beatmap", "*.osu"), ("Minden fájl", "*.*")]))
kimenet_gomb.on_click(lambda _: talloz(kimenet, "Kimeneti mappa", mappa=True))
hatter_gomb.on_click(lambda _: talloz(hatter, "Háttérkép", [("Kép", "*.jpg *.jpeg *.png"), ("Minden fájl", "*.*")]))

betolt()
feladat.observe(alkalmaz_feladatot, names="value")
jatekmod.observe(frissit_cimkek, names="value")
minden_cimke.observe(frissit_cimkek, names="value")
hozzaad.observe(lambda _: zarolas(), names="value")

urlap = widgets.VBox([
    widgets.HTML("<h2 style='margin:0 0 8px 0'>Beatmap generálása</h2>"),
    widgets.HTML("<p style='margin:0 0 8px 0'>A mesterséges intelligencia használatát a beatmapben mindig jelezd. A generálás csak a négyzet bejelölése után indul.</p>"),
    elfogadom,
    mezo("Feladat", feladat),
    feladat_szoveg,
    widgets.HTML("<h3>Fájlok</h3>"),
    mezo("Hang", widgets.HBox([hang, hang_gomb]), "Ha a .osu mellett ott a dal, ez üresen maradhat."),
    mezo("Referencia .osu", widgets.HBox([beatmap, beatmap_gomb]), "Stable: File → Open Song Folder. Lazer: File → Edit Externally. Kell in-context használatához és egy szakasz újraírásához."),
    mezo("Kimeneti mappa", widgets.HBox([kimenet, kimenet_gomb])),
    widgets.HTML("<h3>Stílus</h3>"),
    sor(
        mezo("Modell", modell, "A V32 a legjobb. A V30 nem tud más játékmódot, évet, címkét vagy in-contextet."),
        mezo("Játékmód", jatekmod),
    ),
    sor(
        mezo("Csillagérték", nehezseg, "−1: ismeretlen. A dal és a többi beállítás miatt eltérhet."),
        mezo("Mapper azonosító", mapper_id, "A profil URL-jében lévő szám. −1: nincs."),
        mezo("Év", ev, "2007 és 2024 között érdemes megadni. −1: ismeretlen."),
    ),
    mezo("LoRA", lora, "Opcionális Hugging Face repo vagy helyi mappa."),
    minden_cimke,
    widgets.HTML("<b>Címkék</b> <span style='color:#555'>– üresen a referencia címkéi maradnak, új mapnál a modell dönt. <a href='https://osu.ppy.sh/wiki/en/Beatmap/Beatmap_tags'>osu! wiki</a></span>"),
    sor(*[mezo(f"{i}. címke", doboz) for i, doboz in enumerate(pozitiv, start=1)]),
    widgets.HTML("<b>Negatív címkék</b> <span style='color:#555'>– ami ne legyen a mapban. Csak cfg &gt; 1 mellett hatnak, és legyen belőlük ugyanannyi, mint a rendes címkéből.</span>"),
    sor(*[mezo(f"{i}. negatív", doboz) for i, doboz in enumerate(negativ, start=1)]),
    widgets.Accordion(children=[
        widgets.VBox([
            sor(
                mezo("HP", hp, "−1: automatikus"),
                mezo("CS", cs, "Mania kulcsszámát a kulcsok mező adja."),
                mezo("OD", od),
                mezo("AR", ar),
                mezo("SV", sv, "Általában 0.4–3.6."),
                mezo("Slider tick", slider_tick),
            ),
            sor(
                mezo("Mania kulcsok", billentyu, "Csak mania."),
                mezo("Hold arány", hold, "0–1, csak mania. −1: automatikus."),
                mezo("Scroll arány", scroll, "0–1, mania és taiko. −1: automatikus."),
            ),
            hitsounded,
        ]),
        widgets.VBox([
            sor(mezo("Kontextus", kontextus, "TIMING kihagyja a timing generálását. KIAI a kiai időket. MAP a köröket. GD egy másik nehézség ritmusa. NO_HS csak hitsoundot tesz a kész objektumokra."), mezo("Kimenet", kimenet_tipus, "Mindent generál, vagy csak timinget.")),
            hozzaad,
            osz,
            felulir,
            sor(
                mezo("Kezdet (ms)", kezdet, "−1: a dal eleje."),
                mezo("Vége (ms)", veg, "−1: a dal vége."),
            ),
            sor(
                mezo("Cfg", cfg, "Nagyobb érték jobban követi a címkéket és a mapper stílust. A magas cfg váratlan eredményt adhat."),
                mezo("Hőmérséklet", homerseklet, "Alacsonyabb érték óvatosabb. Rövid szakasznál érdemes kicsit csökkenteni."),
                mezo("Seed", seed, "−1: véletlen. Ugyanaz a seed ugyanazt a mapot próbálja újra."),
            ),
            mezo("Timing tűrés (ms)", timing_tures, "Nagyobb érték kevesebb timing pontot hagy."),
            super_timing,
            sor(
                mezo("Szuper timing sugarak", sugarak, "Több sugár kicsit pontosabb, és lassabb."),
                mezo("Szuper timing ismétlések", iteraciok, "Több ismétlés kicsit pontosabb, és lassabb."),
                mezo("BPM-váltás küszöbe", bpm_kuszob, "Nagyobb érték kevesebb BPM-váltást enged."),
            ),
        ]),
        widgets.VBox([
            sor(
                mezo("Cím", cim),
                mezo("Előadó", eloado),
                mezo("Készítő", keszito),
                mezo("Nehézség neve", verzio),
                mezo("Forrás", forras),
            ),
            mezo("Háttérkép", widgets.HBox([hatter, hatter_gomb])),
        ]),
    ], titles=("Nehézségi számok", "Generálás", "Metaadatok"), selected_index=None),
    widgets.HTML("<div style='height:8px'></div>"),
    gomb,
    naplo,
])
display(urlap)
